# JaamSim Log Analysis: Airport Departure Terminal

Notebook ini membaca output JaamSim (log + file `.rep`) lalu menghasilkan tabel statistik dan grafik untuk report.

**Cara pakai di laptop (VS Code / Jupyter)**
1. `pip install -r requirements.txt` dari root repo.
2. Pilih `MODEL` di section 2: `A_existing`, `B_proposed_1`, atau `C_proposed_2`.
3. `Run All`. Log dan `.rep` dibaca dari `../models/<MODEL>/`, grafik tersimpan di `figures/<MODEL>/` (perbandingan di `figures/comparison/`).

**Cara pakai di Google Colab**
1. Pilih `MODEL` di section 2, lalu `Runtime > Run all`.
2. Di cell **Upload**, pilih 4 file: `<SYSTEM>-ArrivalLogger.log`, `<SYSTEM>-PassengerLogger.log`, `<SYSTEM>-QueueLogger.log`, `<SYSTEM>.rep`. Untuk section 11 boleh upload file ketiga model sekaligus; file otomatis dipindah ke folder model sesuai prefix namanya.
3. Semua grafik bisa di-download sebagai zip di cell terakhir.

**Sebelum commit:** `Clear All Outputs` lalu save, supaya notebook tidak conflict di Git.

## 1. Setup

In [ ]:
import os, re, glob, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 200, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#999999', 'axes.labelcolor': '#333333',
    'xtick.color': '#555555', 'ytick.color': '#555555',
})
COLOR = {1: '#2a78d6', 2: '#eb6834'}          # 1 = Economy, 2 = Priority
CLASS_NAME = {1: 'Economy', 2: 'Priority'}
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300']

## 2. Konfigurasi
Ubah `MODEL` sesuai folder model yang mau dianalisis; `SYSTEM` (nama cfg) terisi otomatis dari `MODELS`. Parameter teori dipakai untuk membandingkan simulasi dengan perhitungan analitik.

In [ ]:
MODELS = {                       # folder model -> nama cfg tanpa .cfg
    'A_existing':   'Existing_System',
    'B_proposed_1': 'Proposed_System_1',
    'C_proposed_2': 'Proposed_System_2',
}
MODEL = 'A_existing'             # pilih: 'A_existing', 'B_proposed_1', 'C_proposed_2'
SYSTEM = MODELS[MODEL]
RUN_MIN = 240                    # durasi run (menit)
MEAN_IA_S = 12                   # mean inter-arrival (detik)
P_ECO, P_PRIO = 0.7, 0.3

# Mean service time teoretis (menit): Triangular mean = (a+m+b)/3
ES = {
    'Eco check-in':  (1.5 + 2.5 + 4.5) / 3,
    'Prio check-in': (1.0 + 1.5 + 2.5) / 3,
    'Security':      (0.8 + 1.2 + 2.0) / 3,
    'Gate':          0.4,
}

# Lokasi folder (relatif terhadap notebook)
if IN_COLAB:
    MODELS_DIR, FIG_ROOT = 'models', 'figures'
elif os.path.isdir('../models'):         # normal: working dir = folder analysis/
    MODELS_DIR, FIG_ROOT = '../models', 'figures'
elif os.path.isdir('models'):            # VS Code diset jalan dari root repo
    MODELS_DIR, FIG_ROOT = 'models', 'analysis/figures'
else:
    raise FileNotFoundError('Folder models/ tidak ketemu. Jalankan notebook dari folder analysis/.')

MODEL_DIR = f'{MODELS_DIR}/{MODEL}'
FIG_DIR = f'{FIG_ROOT}/{MODEL}'
CMP_DIR = f'{FIG_ROOT}/comparison'
for d in [MODEL_DIR, FIG_DIR, CMP_DIR]:
    os.makedirs(d, exist_ok=True)

## 3. Upload file (khusus Colab)

In [ ]:
if IN_COLAB:
    up = files.upload()
    for name in up:
        # Colab menambah ' (1)' kalau nama file sama; rapikan, lalu pindah ke folder modelnya
        clean = re.sub(r' \(\d+\)(?=\.)', '', name)
        for m, s in MODELS.items():
            if clean.startswith((f'{s}-', f'{s}.')):
                os.makedirs(f'{MODELS_DIR}/{m}', exist_ok=True)
                shutil.move(name, f'{MODELS_DIR}/{m}/{clean}')
                break
        else:
            print('Nama file tidak dikenali:', name)
print(MODEL_DIR, sorted(f for f in os.listdir(MODEL_DIR) if f.startswith(SYSTEM)))

## 4. Fungsi pembaca file

In [ ]:
def read_log(path):
    """Baca log JaamSim: lewati metadata sampai baris kosong, lalu header + data."""
    with open(path, encoding='utf-8', errors='replace') as f:
        lines = f.read().splitlines()
    start = next(i for i, l in enumerate(lines) if l.strip() == '') + 1
    header = lines[start].split('\t')
    rows = [l.split('\t') for l in lines[start + 1:] if l.strip()]
    df = pd.DataFrame(rows, columns=header)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce') if c != 'this.obj' else df[c]
    return df

def read_rep(path):
    """Baca .rep JaamSim jadi tabel: object | output | value | unit."""
    rows, section = [], None
    with open(path, encoding='utf-8', errors='replace') as f:
        for l in f.read().splitlines():
            if l.startswith('***'):
                section = l.strip('* ').strip(); continue
            p = l.split('\t')
            if len(p) == 4 and section:
                rows.append([section, p[0], p[1], pd.to_numeric(p[2], errors='coerce'), p[3]])
    return pd.DataFrame(rows, columns=['class', 'object', 'output', 'value', 'unit'])

def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/{name}.png', bbox_inches='tight')
    plt.show()

f_arr = f'{MODEL_DIR}/{SYSTEM}-ArrivalLogger.log'
f_pax = f'{MODEL_DIR}/{SYSTEM}-PassengerLogger.log'
f_q   = f'{MODEL_DIR}/{SYSTEM}-QueueLogger.log'
f_rep = f'{MODEL_DIR}/{SYSTEM}.rep'
have = {k: os.path.exists(v) for k, v in [('arrival', f_arr), ('passenger', f_pax), ('queue', f_q), ('rep', f_rep)]}
print(have)

## 5. Load data

In [ ]:
arr = read_log(f_arr) if have['arrival'] else None
if arr is not None:
    arr = arr.iloc[:, [0, 2]]
    arr.columns = ['t', 'cls']

pax = None
if have['passenger']:
    raw = read_log(f_pax)
    if raw.shape[1] >= 11:
        pax = raw.iloc[:, [0, 2, 3, 4, 5, 6, 7, 8, 9, 10]].copy()
        pax.columns = ['t_dep', 'cls', 't_arr', 'cycle', 'walk_checkin', 'checkin',
                       'walk_security', 'security', 'walk_lounge', 'gate']
    else:   # format lama: cuma kelas + cycle time
        pax = raw.iloc[:, [0, 2, raw.shape[1] - 1]].copy()
        pax.columns = ['t_dep', 'cls', 'cycle']
        pax['t_arr'] = pax['t_dep'] - pax['cycle']

q = None
if have['queue']:
    q = read_log(f_q)
    q.columns = ['t'] + [re.sub(r'^\[(.+?)\]\.(.+)$', r'\1.\2', c) for c in q.columns[1:]]
    q = q.dropna(axis=1, how='all')

rep = read_rep(f_rep) if have['rep'] else None
for n, d in [('arrival', arr), ('passenger', pax), ('queue', q), ('rep', rep)]:
    print(n, None if d is None else d.shape)

## 6. Verifikasi input
Uji proporsi kelas (chi-square goodness-of-fit) dan distribusi inter-arrival (Kolmogorov-Smirnov vs Exponential).

In [ ]:
if arr is not None:
    n = len(arr); n_eco = (arr.cls == 1).sum(); n_prio = (arr.cls == 2).sum()
    chi = stats.chisquare([n_eco, n_prio], [n * P_ECO, n * P_PRIO])
    ia = np.diff(arr.t.values) * 60
    ks = stats.kstest(ia, 'expon', args=(0, MEAN_IA_S))
    exp_n = RUN_MIN * 60 / MEAN_IA_S
    ver = pd.DataFrame([
        ['Jumlah kedatangan', f'{exp_n:.0f}', f'{n}', f'z = {(n - exp_n) / np.sqrt(exp_n):.2f}'],
        ['Proporsi Eco / Prio', f'{P_ECO:.0%} / {P_PRIO:.0%}', f'{n_eco/n:.1%} / {n_prio/n:.1%}',
         f'chi2 = {chi.statistic:.3f}, p = {chi.pvalue:.3f}'],
        ['Mean inter-arrival (s)', f'{MEAN_IA_S}', f'{ia.mean():.2f}', f'KS D = {ks.statistic:.3f}, p = {ks.pvalue:.3f}'],
    ], columns=['Uji', 'Teori', 'Simulasi', 'Statistik'])
    display(ver)
    print('Kesimpulan: lolos jika p > 0.05 (tidak berbeda signifikan dari input).')

    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    ax.hist(ia, bins=40, density=True, color=SERIES[0], edgecolor='white', lw=0.6, label='Simulasi')
    x = np.linspace(0, ia.max(), 200)
    ax.plot(x, stats.expon.pdf(x, scale=MEAN_IA_S), color='#333333', lw=2, label=f'Exponential(mean {MEAN_IA_S} s)')
    ax.set_xlabel('Inter-arrival time (s)'); ax.set_ylabel('Density')
    ax.set_title(f'Inter-arrival distribution ({SYSTEM})', loc='left')
    ax.legend(frameon=False)
    savefig(fig, 'inter_arrival_hist')

## 7. Utilisasi dan antrian (dari file .rep)

In [ ]:
if rep is not None:
    srv = rep[(rep['class'] == 'Server') & (rep.output.isin(['Utilisation', 'NumberProcessed']))] \
        .pivot(index='object', columns='output', values='value')
    display(srv.style.format({'Utilisation': '{:.1%}', 'NumberProcessed': '{:.0f}'}))

    qs = rep[(rep['class'] == 'Queue') & (rep.output.isin(
        ['NumberAdded', 'NumberProcessed', 'QueueLengthAverage', 'QueueLengthMaximum', 'AverageQueueTime']))] \
        .pivot(index='object', columns='output', values='value')
    qs['WIP_end'] = qs['NumberAdded'] - qs['NumberProcessed']
    display(qs.round(2))

    dep = rep[(rep.object == 'Departed') & (rep.output == 'NumberAdded')].value.sum()
    gen = rep[(rep['class'] == 'EntityGenerator') & (rep.output == 'NumberAdded')].value.sum()
    print(f'Throughput: {dep:.0f} dari {gen:.0f} pax selesai ({dep/gen:.1%})')

    fig, ax = plt.subplots(figsize=(7, 0.45 * len(srv) + 1.2))
    s = srv.Utilisation.sort_values()
    bars = ax.barh(s.index, s.values, color=[SERIES[1] if v >= 0.9 else SERIES[0] for v in s.values], height=0.6)
    for b, v in zip(bars, s.values):
        ax.text(v + 0.01, b.get_y() + b.get_height() / 2, f'{v:.1%}', va='center', fontsize=9, color='#333333')
    ax.axvline(0.85, color='#999999', ls='--', lw=1); ax.text(0.85, len(s) - 0.4, ' 85%', color='#666666', fontsize=8)
    ax.set_xlim(0, 1.12); ax.set_xlabel('Utilisation')
    ax.set_title(f'Server utilisation ({SYSTEM}), orange = 90% or more', loc='left')
    savefig(fig, 'server_utilisation')

## 8. Panjang antrian vs waktu (QueueLogger)
Antrian yang naik linear terus = stasiun tidak stabil.

In [ ]:
if q is not None:
    qcols = [c for c in q.columns if c.endswith('QueueLength')]
    fig, ax = plt.subplots(figsize=(7.5, 4))
    rows = []
    for i, c in enumerate(qcols):
        name = c.replace('.QueueLength', '')
        ax.plot(q.t, q[c], lw=2, color=SERIES[i % len(SERIES)], label=name)
        sl = stats.linregress(q.t, q[c])
        rows.append([name, q[c].mean(), q[c].max(), q[c].iloc[-1], sl.slope, sl.rvalue ** 2])
    ax.set_xlabel('Simulation time (min)'); ax.set_ylabel('Passengers in queue')
    ax.set_title(f'Queue length over time ({SYSTEM})', loc='left')
    ax.grid(axis='y', color='#e6e6e6'); ax.set_axisbelow(True)
    ax.legend(frameon=False, fontsize=8)
    savefig(fig, 'queue_length')
    display(pd.DataFrame(rows, columns=['Queue', 'Mean', 'Max', 'End', 'Growth (pax/min)', 'R2']).round(3))

    flow = [c for c in q.columns if c.endswith('NumberAdded')]
    if len(flow) >= 2:
        fig, ax = plt.subplots(figsize=(7.5, 3.6))
        for i, c in enumerate(flow):
            ax.plot(q.t, q[c], lw=2, color=SERIES[i], label=c.replace('.NumberAdded', ''))
        ax.fill_between(q.t, q[flow[1]], q[flow[0]], color='#dddddd', alpha=0.6, label='WIP (in system)')
        ax.set_xlabel('Simulation time (min)'); ax.set_ylabel('Cumulative passengers')
        ax.set_title(f'Cumulative arrivals vs departures ({SYSTEM})', loc='left')
        ax.legend(frameon=False, fontsize=8)
        savefig(fig, 'cumulative_flow')

## 9. Cycle time (PassengerLogger)
Catatan: hanya pax yang sudah sampai gate yang tercatat (survivorship bias kalau sistem overload).

In [ ]:
if pax is not None:
    rows = []
    fig, ax = plt.subplots(figsize=(7.5, 4))
    for k in [1, 2]:
        m = pax[pax.cls == k]
        if len(m) < 3: continue
        sl = stats.linregress(m.t_arr, m.cycle)
        se = m.cycle.std(ddof=1) / np.sqrt(len(m))
        h = stats.t.ppf(0.975, len(m) - 1) * se
        rows.append([CLASS_NAME[k], len(m), m.cycle.mean(), h, m.cycle.std(ddof=1), m.cycle.max(),
                     sl.slope, sl.rvalue ** 2, np.diff(np.sort(m.t_dep)).mean()])
        ax.scatter(m.t_arr, m.cycle, s=14, color=COLOR[k], alpha=0.8, edgecolor='white', lw=0.5,
                   label=f'{CLASS_NAME[k]} (slope {sl.slope:.2f}, R² {sl.rvalue**2:.3f})')
        x = np.array([m.t_arr.min(), m.t_arr.max()])
        ax.plot(x, sl.intercept + sl.slope * x, color=COLOR[k], lw=1.5)
    ax.set_xlabel('Arrival time (min)'); ax.set_ylabel('Cycle time (min)')
    ax.set_title(f'Cycle time vs arrival time ({SYSTEM})', loc='left')
    ax.grid(axis='y', color='#e6e6e6'); ax.set_axisbelow(True)
    ax.legend(frameon=False, loc='best')
    savefig(fig, 'cycle_time_vs_arrival')
    ct = pd.DataFrame(rows, columns=['Class', 'n', 'Mean CT', '95% CI (±)', 'SD', 'Max',
                                     'Slope vs arrival', 'R2', 'Mean inter-departure'])
    display(ct.round(3))
    print('Slope ~ 0 = stabil. Slope > 0 dengan R2 tinggi = antrian terus menumpuk.')

## 10. Breakdown waktu per tahap

In [ ]:
stages = ['walk_checkin', 'checkin', 'walk_security', 'security', 'walk_lounge', 'gate']
if pax is not None and all(s in pax.columns for s in stages):
    br = pax.groupby('cls')[stages + ['cycle']].mean().rename(index=CLASS_NAME)
    share = (br[stages].div(br['cycle'], axis=0) * 100).round(1)
    display(br.round(2)); display(share.add_suffix(' %'))

    labels = ['Walk to check-in', 'Check-in', 'Walk to security', 'Security', 'Walk to lounge', 'Gate']
    fig, ax = plt.subplots(figsize=(7.5, 3.3))
    left = np.zeros(len(br))
    for i, (s, lab) in enumerate(zip(stages, labels)):
        ax.barh(br.index, br[s], left=left, color=SERIES[i], height=0.55, label=lab, edgecolor='white', lw=1.5)
        left += br[s].values
    for y, tot in enumerate(left):
        ax.text(tot + 1, y, f'{tot:.1f} min', va='center', fontsize=9, color='#333333')
    ax.set_xlabel('Mean time (min)'); ax.set_xlim(0, left.max() * 1.15)
    ax.set_title(f'Where passengers spend their time ({SYSTEM})', loc='left')
    ax.legend(frameon=False, fontsize=8, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.42))
    savefig(fig, 'stage_breakdown')

    # Waktu di tahap vs E[S]: selisihnya = waktu tunggu rata-rata
    cmp = pd.DataFrame({
        'Stage': ['Eco check-in', 'Prio check-in', 'Eco security', 'Prio security', 'Gate (all)'],
        'Simulated time in stage': [pax[pax.cls == 1].checkin.mean(), pax[pax.cls == 2].checkin.mean(),
                                    pax[pax.cls == 1].security.mean(), pax[pax.cls == 2].security.mean(), pax.gate.mean()],
        'E[S] theory': [ES['Eco check-in'], ES['Prio check-in'], ES['Security'], ES['Security'], ES['Gate']],
    })
    cmp['Implied waiting'] = cmp['Simulated time in stage'] - cmp['E[S] theory']
    display(cmp.round(3))

## 11. Perbandingan antar sistem (opsional)
Isi `COMPARE` dengan folder model yang mau dibandingkan. Log dan `.rep` dibaca dari folder masing-masing model; model yang belum punya log otomatis di-skip.

In [ ]:
COMPARE = ['A_existing', 'B_proposed_1', 'C_proposed_2']

summ = []
for model in COMPARE:
    sysname = MODELS[model]
    fp = f'{MODELS_DIR}/{model}/{sysname}-PassengerLogger.log'; fr = f'{MODELS_DIR}/{model}/{sysname}.rep'
    if not os.path.exists(fp): print('skip', sysname); continue
    d = read_log(fp)
    d = d.iloc[:, [2, 4]]; d.columns = ['cls', 'cycle']
    r = read_rep(fr) if os.path.exists(fr) else None
    done = r[(r.object == 'Departed') & (r.output == 'NumberAdded')].value.sum() if r is not None else np.nan
    gen = r[(r['class'] == 'EntityGenerator') & (r.output == 'NumberAdded')].value.sum() if r is not None else np.nan
    for k in [1, 2]:
        c = d[d.cls == k].cycle
        summ.append([sysname, CLASS_NAME[k], len(c), c.mean(), c.std(ddof=1), 100 * done / gen])
summ = pd.DataFrame(summ, columns=['System', 'Class', 'n', 'Mean CT', 'SD', 'Throughput %'])
display(summ.round(3))

if len(summ):
    fig, ax = plt.subplots(figsize=(7, 3.8))
    systems = summ.System.unique(); w = 0.38; x = np.arange(len(systems))
    for j, k in enumerate([1, 2]):
        v = summ[summ.Class == CLASS_NAME[k]].set_index('System').reindex(systems)['Mean CT']
        b = ax.bar(x + (j - 0.5) * w, v, w * 0.95, color=COLOR[k], label=CLASS_NAME[k])
        for xi, vi in zip(x + (j - 0.5) * w, v):
            ax.text(xi, vi, f'{vi:.1f}', ha='center', va='bottom', fontsize=8, color='#333333')
    ax.set_xticks(x); ax.set_xticklabels(systems)
    ax.set_ylabel('Mean cycle time (min)'); ax.set_title('Mean cycle time by system', loc='left')
    ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(f'{CMP_DIR}/mean_cycle_time.png', bbox_inches='tight'); plt.show()

## 12. Daftar grafik (dan download zip di Colab)

In [ ]:
print('Grafik tersimpan:', sorted(glob.glob(f'{FIG_ROOT}/**/*.png', recursive=True)))
if IN_COLAB:
    shutil.make_archive('figures', 'zip', FIG_ROOT)
    files.download('figures.zip')